# Module 11: Control for Baseline Ratios and Prepare Filing Review Cases

目标：检查 Module 10 的关联是否受本期比率、公司差异及年度因素影响，并生成下一模块的关键个案核对清单。
仍为 8 个 Code Cells，中文逐格讲解，英文代码、注释、列名、图表和文件名。
使用 statsmodels 的公式接口，让你不用手工建立虚拟变量或写矩阵运算；分析代码没有自定义函数。

接在现有 Notebook 的 Module 10 后，从 Cell 1 到 Cell 8 依次运行；也可单独运行 Guide，使用此前相同目录。
输入：Module 9 的两个结果配对 CSV、Module 10 metadata，以及 candidate_scope_review.csv。
本模块不下载年报；Cell 7 生成 SEC 文件目录链接，供 Module 12 查报表和附注。
输出：模型结果、预定敏感性结果、基期等价检查、一张系数图、个案队列、人工核对工作表和处理说明。

**收尾安排：包含本模块在内，预计还剩 3 个模块。**
- Module 11：基期控制与回归诊断，准备核对个案。
- Module 12：核对关键个案与公司资格，记录会计口径问题，确定报告使用的样本与限制；发现问题则修正上游并重跑。
- Module 13：整理最终 report、图表、方法说明与可复现材料。

这不是承诺 7 个个案能审核完 42 家公司的全部历史。未完成的公司/年度审核必须明确标注，必要时缩小最终样本或把结果称为暂定研究。
150 条缺失指标建议也不需要全部填满；有证据且符合定义的项目才采用，其他缺失可以保留并说明样本覆盖影响。

本模块是回顾性的条件关联分析，不进行因果识别、样本外预测或股价回测。
聚类标准误和更多控制项不能解决未核对财务口径、样本选择或披露时间问题。
按公司聚类，不同时按年份聚类；年份控制也不等于消除了所有跨公司共同冲击。
不同公司历史长度不一，基期变量与公司固定效应组合在较短面板中也可能有偏，回归仍用于探索。


## Cell 1: Import the Regression Tools

沿用 pandas、matplotlib，新增 numpy 与 statsmodels.formula.api。
np.log 用于把资产规模转换为自然对数；smf.ols 用简短公式建立普通最小二乘回归。
你的本地环境已有 statsmodels，验证版本为 0.14.6。若 Notebook 的 Kernel 提示缺包，另开一格运行 `%pip install statsmodels`。
不需要为了运行本模块升级已有环境。

复用 PROJECT_ROOT；路径不对时，在导入语句后设置 `PROJECT_ROOT = Path("[project root]")`。
FIGURE_DIR 保存系数图，mkdir 允许目录已存在。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels
import statsmodels.formula.api as smf
from IPython.display import display

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FIGURE_DIR = PROJECT_ROOT / "outputs" / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
print("Project directory:", PROJECT_ROOT)
print("Statsmodels version:", statsmodels.__version__)


## Cell 2: Prepare the Two Regression Samples

重新从已保存的 Module 9 文件读数据，不依赖 Kernel 中 Module 10 留下的临时变量。
settings 中每个名称对应文件名、本期比率和下一期变化列；普通循环分别准备毛利与现金流表。
inventory_gap_pp、outcome_change_pp、current_ratio_pct、next_ratio_pct 的单位与 Module 10 一致。
revenue_growth_pp 是收入增长率乘 100；log_assets 是本期 total_assets 的自然对数，减少公司规模数量级差异。
Module 9 已要求资产为正，因此这里可以取对数；没有用下一年资产或下一年收入作为本期解释变量。

target_end_year 使用下一年度实际终点的日历年，后面作为分类控制，而不是公司财年名称。
由于零售公司年末日期不同，这只是粗略年度控制，不保证这些记录覆盖相同的经营时段。

assert 防止回归工具悄悄删除缺失输入，让不同模型在不知情的情况下使用不同样本。
common_keys 按公司和本期年度终点连接，保留两个结果都有合格数据的 399 个键。
company_review 仅用于取得 CIK 和保留当前审核状态；本模块不会把 pending 自动改为已审核。
应打印毛利 401 个 / 36 家，现金流 490 个 / 42 家，共同键 399 个。


In [ ]:
settings = {
    "Gross margin": ("expanded_gm_next_year_pairs.csv", "gross_margin", "next_gross_margin_change"),
    "OCF-to-revenue": ("expanded_ocf_next_year_pairs.csv", "ocf_margin", "next_ocf_margin_change"),
}
regression_tables = {}
for name, (filename, ratio, outcome) in settings.items():
    table = pd.read_csv(PROCESSED_DIR / filename, parse_dates=["period_end", "next_period_end"])
    table["inventory_gap_pp"] = table["inventory_sales_gap"] * 100
    table["current_ratio_pct"] = table[ratio] * 100
    table["next_ratio_pct"] = table["next_" + ratio] * 100
    table["outcome_change_pp"] = table[outcome] * 100
    table["revenue_growth_pp"] = table["revenue_growth"] * 100
    table["log_assets"] = np.log(table["total_assets"])
    table["target_end_year"] = table["next_period_end"].dt.year
    inputs = ["inventory_gap_pp", "current_ratio_pct", "next_ratio_pct", "outcome_change_pp", "revenue_growth_pp", "log_assets", "target_end_year"]
    assert np.isfinite(table[inputs].to_numpy()).all(), "Review missing or invalid regression inputs."
    assert not table.duplicated(["ticker", "period_end"]).any(), "Duplicate company-year pairs."
    regression_tables[name] = table
    print(name, "pairs:", len(table), "| companies:", table["ticker"].nunique())
common_keys = regression_tables["Gross margin"][["ticker", "period_end"]].merge(
    regression_tables["OCF-to-revenue"][["ticker", "period_end"]], on=["ticker", "period_end"], validate="one_to_one",
)
company_review = pd.read_csv(PROCESSED_DIR / "candidate_scope_review.csv", dtype={"cik10": "string"}, keep_default_na=False)
module10_metadata = json.loads((PROCESSED_DIR / "expanded_module10_metadata.json").read_text(encoding="utf-8"))
print("Common outcome pairs:", len(common_keys))


## Cell 3: Fit Five Clearly Defined Specifications

公式字符串里，`~` 左边是结果变量，右边是解释变量；`+` 表示新增控制项。
公式默认包含截距，不用自己加一列全为 1 的数。

固定比较五种设定：
1. Change: unadjusted：下一年变化只与库存差关联，延续 Module 10 的问题。
2. Next level: unadjusted：直接看下一年比率水平，区分变化与水平。
3. Next level: baseline：在相同本期比率条件下，看库存差与下一年水平的关联。
4. Change: baseline：使用同样解释变量改为变化目标，检查数学等价关系；不是一份独立验证。
5. Next level: company/year：再控制本期收入增长、公司规模、公司固定差异与目标年度分类。

`C(ticker)` 和 `C(target_end_year)` 让公式工具建立分类虚拟变量。
公司固定效应控制样本中公司不随年份变化的平均差异；年度项控制同一终点日历年记录的平均差异。
有公司固定效应时，不再额外放 C(subgroup)，因为子行业已嵌套在公司里。
也不同时放库存增长、收入增长和两者之差，否则三者存在完全代数重复。

smf.ols 先建立模型，fit 再估计系数。
`cov_type="cluster"` 按公司计算不确定性，允许同一公司不同年度的误差相关；它不改变 OLS 点估计。
groups 指明公司归属，两个 correction 设置保留小样本修正；use_t=True 使用聚类数减 1 的 t 分布自由度构造区间。
这不能保证有限样本区间精确，也没有同时处理全部公司之间的相关性。
models 用 (结果名称, 模型名称) 作为字典键，便于下一格取回完整估计结果。
eval_env=-1 使用干净公式环境，避免 Notebook 中碰巧存在同名 C 变量干扰分类函数。

参考：[公式接口](https://www.statsmodels.org/stable/example_formulas.html)、[OLS.fit](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.OLS.fit.html)、[公司聚类与自由度修正](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.get_robustcov_results.html)。


In [ ]:
model_specs = {
    "Change: unadjusted": "outcome_change_pp ~ inventory_gap_pp",
    "Next level: unadjusted": "next_ratio_pct ~ inventory_gap_pp",
    "Next level: baseline": "next_ratio_pct ~ inventory_gap_pp + current_ratio_pct",
    "Change: baseline": "outcome_change_pp ~ inventory_gap_pp + current_ratio_pct",
    "Next level: company/year": "next_ratio_pct ~ inventory_gap_pp + current_ratio_pct + revenue_growth_pp + log_assets + C(ticker) + C(target_end_year)",
}
models = {}
for name, table in regression_tables.items():
    for model_name, formula in model_specs.items():
        model = smf.ols(formula, data=table, eval_env=-1).fit(
            cov_type="cluster", cov_kwds={"groups": table["ticker"], "use_correction": True, "df_correction": True}, use_t=True,
        )
        models[(name, model_name)] = model
print("Main model fits:", len(models))


## Cell 4: Read the Inventory Coefficient and Check Baseline Equivalence

model.params 取回归系数；conf_int().loc["inventory_gap_pp"] 取库存系数的 95% 区间两端。
gap_coefficient 表示库存增长差高 1 个百分点时，对应的结果差异，单位为百分点。
为便于解释，association_10pp = gap_coefficient × 10；区间两端也乘 10。
例如 association_10pp=-0.20 表示本模型中库存差高 10 个百分点对应较低 0.20 个百分点的结果。
这是模型内条件关联，不是干预效果、单家公司预测或股价收益。

不同模型的结果变量不一样：Change 模型对应变化，Next level 模型对应水平，不能仅按系数大小评优。
baseline_coefficient 是基期比率系数，未放这个变量的模型留 NaN。
r_squared 是样本内拟合度。比率本来高度持续、或加入很多公司虚拟变量，都可能让它很高；不表示已经有样本外预测能力。
parameters 记录估计参数数目，companies 才是聚类数量，不是把 401 / 490 行当成独立公司。

**为什么第 3、4 个模型会给出相同的库存系数？**
因为 change = next level - current level，且两个模型右侧都包含 current level。
在同一批数据与相同解释变量下，改写结果变量只会让 current level 的系数减少 1；库存系数与残差不变。
identity_check 中：库存系数差应接近 0，基期系数差加 1 也应接近 0。
极小浮点误差正常，不能把这种重复结果当作两次独立支持。

这里展示置信区间，不输出 p 值或星号。区间跨零表示该模型和误差假设下方向尚不精确，不等于证明不存在任何关联。
参考：[conf_int](https://www.statsmodels.org/stable/generated/statsmodels.regression.linear_model.RegressionResults.conf_int.html)。


In [ ]:
model_rows = []
for (name, model_name), model in models.items():
    interval = model.conf_int().loc["inventory_gap_pp"]
    model_rows.append({
        "outcome": name, "model": model_name, "pairs": int(model.nobs), "companies": regression_tables[name]["ticker"].nunique(),
        "parameters": len(model.params), "gap_coefficient": model.params["inventory_gap_pp"],
        "association_10pp": model.params["inventory_gap_pp"] * 10,
        "ci_lower_10pp": interval.iloc[0] * 10, "ci_upper_10pp": interval.iloc[1] * 10,
        "baseline_coefficient": model.params.get("current_ratio_pct", float("nan")), "r_squared": model.rsquared,
    })
model_results = pd.DataFrame(model_rows)
identity_rows = []
for name in regression_tables:
    level_model = models[(name, "Next level: baseline")]
    change_model = models[(name, "Change: baseline")]
    identity_rows.append({
        "outcome": name,
        "gap_coefficient_difference": change_model.params["inventory_gap_pp"] - level_model.params["inventory_gap_pp"],
        "baseline_difference_plus_one": change_model.params["current_ratio_pct"] - level_model.params["current_ratio_pct"] + 1,
    })
identity_check = pd.DataFrame(identity_rows)
display(model_results.round(4))
display(identity_check.round(10))


## Cell 5: Refit the Adjusted Model on the Predefined Samples

沿用 Module 10 的四种样本规则，只对 Next level: company/year 模型做敏感性检查。
All pairs 直接复用已估计的完整样本模型，不重复拟合。
其余三种分别为年度长度不变、共同结果样本、库存差第 1–99 百分位区间。
修剪边界只按当前库存信号决定，不按下一年结果或估计显著性选择。

在每个子样本内重新估计全部控制系数和公司/年度项，没有拿全样本模型去假装子样本模型。
样本变了，公司权重、有效年度和控制估计也会变，因此方向或区间变化需要一起看，不能只挑一个版本汇报。
年度长度标记包含目标年度信息，此处用于事后审核，不作为当时已知的预测输入。

当前预计四种样本行数：毛利 401 / 201 / 399 / 393；现金流 490 / 235 / 399 / 480。
模型仍使用公司年度 OLS 权重，不是 Module 10 的公司等权分组概要，两者回答的问题不同。


In [ ]:
sensitivity_rows = []
adjusted_formula = model_specs["Next level: company/year"]
for name, table in regression_tables.items():
    lower, upper = table["inventory_gap_pp"].quantile([0.01, 0.99])
    common = table.merge(common_keys, on=["ticker", "period_end"], validate="one_to_one")
    for sample_name, sample in [
        ("All pairs", table), ("Unchanged period lengths", table.loc[~table["pair_period_length_changed"]]),
        ("Common outcome pairs", common), ("Central 98% of inventory gaps", table.loc[table["inventory_gap_pp"].between(lower, upper)]),
    ]:
        if sample_name == "All pairs":
            model = models[(name, "Next level: company/year")]
        else:
            model = smf.ols(adjusted_formula, data=sample, eval_env=-1).fit(
                cov_type="cluster", cov_kwds={"groups": sample["ticker"], "use_correction": True, "df_correction": True}, use_t=True,
            )
        interval = model.conf_int().loc["inventory_gap_pp"]
        sensitivity_rows.append({
            "outcome": name, "sample": sample_name, "pairs": int(model.nobs), "companies": sample["ticker"].nunique(),
            "association_10pp": model.params["inventory_gap_pp"] * 10,
            "ci_lower_10pp": interval.iloc[0] * 10, "ci_upper_10pp": interval.iloc[1] * 10,
            "r_squared": model.rsquared,
        })
regression_sensitivity = pd.DataFrame(sensitivity_rows)
display(regression_sensitivity.round(4))


## Cell 6: Plot Next-Year Level Coefficients and Confidence Intervals

图中只画三个 Next level 模型，保持结果变量同为下一年比率水平；Change 模型留在结果表里。
圆点是库存差高 10 个百分点对应的模型内关联，横线是公司聚类的 95% 系数置信区间。
横轴左侧对应较低下一年比率，右侧对应较高比率；零线帮助读方向。
这是系数区间，不是预测区间，不能解读为单家公司下一年表现的范围。

errorbar 的 xerr 要的是圆点到上下界的距离，而不是直接传入上下界。
positions 是每个模型在纵轴上的行号；set_yticks 把它们改成模型名称。
图上所有预定模型一起展示，不删去跨零的区间；没有显著性星号。
来源和年度终点范围保留在图中，保存为 expanded_baseline_control_coefficients.png。


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 7))
for ax, name in zip(axes, regression_tables):
    chart = model_results.loc[model_results["outcome"].eq(name) & model_results["model"].str.startswith("Next level")]
    positions = list(range(len(chart)))
    distances = [chart["association_10pp"] - chart["ci_lower_10pp"], chart["ci_upper_10pp"] - chart["association_10pp"]]
    ax.errorbar(chart["association_10pp"], positions, xerr=distances, fmt="o", capsize=4, color="tab:blue")
    ax.set_yticks(positions, chart["model"])
    ax.axvline(0, color="gray", linewidth=0.8)
    ax.set_title(f"{name}: {len(regression_tables[name])} pairs")
    ax.set_xlabel("Next-year ratio association per 10 pp larger inventory gap (pp)")
    ax.set_ylabel("Model specification")
first_end = min(table["period_end"].min() for table in regression_tables.values()).date()
last_end = max(table["period_end"].max() for table in regression_tables.values()).date()
fig.text(0.5, 0.01, f"Source: Module 9 SEC-based pairs | Current ends: {first_end} to {last_end}\nFirm-clustered 95% coefficient intervals | Retrospective, provisional associations", ha="center", fontsize=8)
fig.tight_layout(rect=[0, 0.08, 1, 1])
fig.savefig(FIGURE_DIR / "expanded_baseline_control_coefficients.png", dpi=180, bbox_inches="tight")
plt.show()


## Cell 7: Select Seven Filing Review Cases from Current Signals

将两个结果表合并后按 ticker + period_end 去重，得到任一结果可用的公司年度合集。
这里保留 Module 9 两个结果各自的可用标记和原命名变化列，不用统一的 outcome_change_pp 解释两项结果。
当其中一项结果不合格时，其变化列仍留 NaN，不借另一个结果的值补齐。

选取规则仅使用本期库存信号，不按下一年表现、残差或显著性挑案例：
- 每个子行业库存增长差最高的一条，共 5 条，用于审核极端信号与口径。
- 正差值组中最接近库存差中位数的一条，用于补充典型信号。
- 库存差最低的一条，用于对照极端负差值。

最高/最低信号本身不能代表整个行业，也不能证明总体机制；后续不因不符合故事而换案例。
同一个公司年度若被多条规则选中，只保留一次，所以以后不一定恰好 7 条。
当前预计 7 个公司年度，涉及 AKA、ARHS、KSS、SPWH、ULTA、URBN 六家公司；SPWH 有两个年度。

sort_values 按规则排序；drop_duplicates 只用于候选合集与个案清单，不修改原研究表。
idxmin 取得离中位数最近的行号；loc[[行号]] 保留一行 DataFrame，方便 concat。
CIK 从官方身份表读取，accn 去掉横线后组成 SEC 文件目录。打开目录，再选择该次提交的 10-K 正文。
current_filing_url 核对本期增长及两年比较口径，next_filing_url 核对下一年结果；暂不自动下载。

每个案例，下一模块要核对：报表单位、库存及收入范围、收购/IPO/重列、成本或毛利范围、现金流营运资本解释。
会计口径核对与经营原因解释是两件事。review_status=pending；证据 URL 和 review_notes 未核对前留空。
业务解释有证据才填写，不因库存差为正就写“积压”，也不因现金流恢复就写“去库存成功”。


In [ ]:
case_union = pd.concat(regression_tables.values(), ignore_index=True).drop_duplicates(["ticker", "period_end"])
high_cases = case_union.sort_values(["inventory_sales_gap", "ticker", "period_end"], ascending=[False, True, True]).drop_duplicates("subgroup").copy()
high_cases["selection_reason"] = "Largest inventory gap within subgroup"
positive = case_union.loc[case_union["inventory_sales_gap"].gt(0)]
distance = (positive["inventory_sales_gap"] - positive["inventory_sales_gap"].median()).abs()
typical_case = positive.loc[[distance.idxmin()]].copy()
typical_case["selection_reason"] = "Closest to the median positive inventory gap"
negative_case = case_union.sort_values(["inventory_sales_gap", "ticker", "period_end"]).head(1).copy()
negative_case["selection_reason"] = "Most negative inventory gap"
cases = pd.concat([high_cases, typical_case, negative_case], ignore_index=True).drop_duplicates(["ticker", "period_end"])
case_columns = [
    "ticker", "subgroup", "period_end", "next_period_end", "accn", "next_accn", "selection_reason",
    "revenue", "revenue_for_analysis", "revenue_growth", "inventory", "inventory_growth", "inventory_gap_pp",
    "gross_margin", "next_gross_margin", "next_gross_margin_change", "ocf_margin", "next_ocf_margin", "next_ocf_margin_change",
    "gm_pair_usable", "ocf_pair_usable", "derived_fields", "revenue_basis_adjusted",
]
cases = cases[case_columns].copy().reset_index(drop=True)
cases["cik10"] = cases["ticker"].map(company_review.set_index("ticker")["cik10"])
for index, row in cases.iterrows():
    for accession_column, url_column in [("accn", "current_filing_url"), ("next_accn", "next_filing_url")]:
        cases.at[index, url_column] = f"https://www.sec.gov/Archives/edgar/data/{int(row['cik10'])}/{row[accession_column].replace('-', '')}/"
cases["review_status"] = "pending"
for column in ["units_and_scope_notes", "revenue_and_inventory_drivers", "next_margin_and_ocf_drivers", "suggested_action", "current_evidence_url", "next_evidence_url", "review_notes"]:
    cases[column] = ""
print("Selected filing review cases:", len(cases))
display(cases[["ticker", "period_end", "next_period_end", "inventory_gap_pp", "selection_reason", "review_status"]])


## Cell 8: Save Models, Cases and the Module Boundaries

保存三个数值表、一个每次重建的个案队列和 metadata。
另建 expanded_module11_case_review.csv 作为人工填写表：第一次运行创建，后续运行保留已有文件和备注。
所以重跑不会清空你的人工核对记录；若后续更新数据使案例键或金额改变，要在 Module 12 比较新队列与旧人工表，不直接沿用旧结论。
本模块也不读取并应用人工核对决定；回归始终沿用 Module 9 的暂定样本。

metadata 保存公式、标准误与区间规则、结果表单位、案例选择方法以及上游状态。
展示与保存的统计是条件关联，不是预测效果。metadata 中 prediction_validated / accounting_scope_verified / point_in_time_verified 仍为 False。

运行完保存 Notebook。重点核对 Cell 4 的两个等价差接近 0、Cell 5 的行数、Cell 7 的案例键。
下一模块再逐个打开年报核对证据；本模块不要求你现在把整张人工表填完。
完成本模块后，按当前安排还剩 Module 12 与 Module 13 两个模块。


In [ ]:
outputs = {
    "expanded_baseline_regression_results.csv": model_results,
    "expanded_baseline_identity_check.csv": identity_check,
    "expanded_adjusted_regression_sensitivity.csv": regression_sensitivity,
    "expanded_module11_case_queue.csv": cases,
}
for filename, table in outputs.items():
    table.to_csv(PROCESSED_DIR / filename, index=False, date_format="%Y-%m-%d")
review_path = PROCESSED_DIR / "expanded_module11_case_review.csv"
if not review_path.exists():
    cases.to_csv(review_path, index=False, date_format="%Y-%m-%d")
else:
    print("Existing manual case review retained:", review_path.name)
metadata = {
    "processed_at_utc": datetime.now(timezone.utc).isoformat(), "module10_metadata": module10_metadata,
    "statsmodels_version": statsmodels.__version__, "sample_status": "provisional", "model_specs": model_specs,
    "model_weighting": "OLS on company-year pairs; not company-equal weighting",
    "covariance_rule": "Firm-clustered CRV1 with small-sample and cluster-df corrections; t-based 95% coefficient intervals",
    "year_rule": "Categorical calendar year of target period end; not fully aligned operating periods",
    "unit": "Ratio levels in percent; growth gaps and changes in percentage points; coefficients scaled to a 10 pp gap",
    "main_model_rows": len(model_results), "sensitivity_rows": len(regression_sensitivity), "case_count": len(cases),
    "case_selection": "Subgroup maxima, median positive-gap representative and minimum gap; use current signals only",
    "manual_review_decisions_applied": module10_metadata.get("module9_metadata", {}).get("manual_review_decisions_applied", False),
    "confidence_intervals_reported": True, "p_values_reported": False,
    "accounting_scope_verified": False, "point_in_time_verified": False, "prediction_validated": False, "causal_identification": False,
}
(PROCESSED_DIR / "expanded_module11_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Module 11 tables saved in:", PROCESSED_DIR)
print("Module 11 figure saved in:", FIGURE_DIR)
